# Spendly final model evaluation

**Purpose:** produce a clean, reproducible benchmark before any model or
application changes are made.

This notebook:

- uses all synthetic users rather than selecting one convenient user;
- keeps train, validation, and test periods chronological;
- fits every scaler and model using training data only;
- compares LSTM forecasts against Linear Regression, Random Forest,
  previous-period, and four-period moving-average baselines;
- trains Isolation Forest on earlier expense transactions and evaluates it
  on a later untouched period;
- saves a compact JSON results block that can be copied back for review.

**Run instructions:** upload `combined_finance_raw.csv`, then select
**Runtime → Restart session and run all**. Do not run cells out of order.
The data are synthetic and results must not be presented as evidence about
real Kenyan young adults.


In [ ]:
# Install only packages that are missing from the current Colab runtime.
import importlib.util
import subprocess
import sys

requirements = {
    "pandas": "pandas>=2.0",
    "numpy": "numpy>=1.26",
    "sklearn": "scikit-learn>=1.4",
    "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
    "joblib": "joblib>=1.3",
    "tensorflow": "tensorflow>=2.16",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import time
import warnings
import zipfile
from collections import deque
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_curve,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.preprocessing import RobustScaler

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
LOOKBACK = 8
FORECAST_HORIZON = 1
MAX_EPOCHS = 30
BATCH_SIZE = 128
OUTPUT_DIR = Path("spendly_final_output")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("TensorFlow:", tf.__version__)
print("Random seed:", RANDOM_STATE)
print("Forecast design: lookback=8 weeks, horizon=1 week")


## 1. Load and audit the raw CSV

The raw dataframe is retained unchanged. Blank budget and anomaly fields
are expected for rows where no matching budget or controlled anomaly exists.


In [ ]:
configured_path = os.environ.get("SPENDLY_DATA_PATH")
candidates = [
    Path(configured_path) if configured_path else None,
    Path("combined_finance_raw.csv"),
    Path("university_trial_colab") / "combined_finance_raw.csv",
]
csv_path = next((path for path in candidates if path is not None and path.is_file()), None)

if csv_path is None:
    try:
        from google.colab import files
        print("Upload combined_finance_raw.csv")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No CSV was uploaded.")
        csv_path = Path(next(iter(uploaded)))
    except ImportError as exc:
        raise FileNotFoundError(
            "combined_finance_raw.csv was not found. Place it beside the notebook."
        ) from exc

dataset_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
raw = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded: {csv_path}")
print(f"SHA256: {dataset_sha256}")
print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
display(raw.head())


In [ ]:
required_columns = {
    "transaction_id", "user_id", "transaction_timestamp", "amount",
    "currency", "category", "transaction_type", "merchant",
    "payment_method", "is_recurring", "user_age", "user_monthly_income",
    "user_employment_status", "budget_amount", "is_anomaly", "anomaly_type",
}
missing_required = sorted(required_columns - set(raw.columns))
if missing_required:
    raise ValueError(f"CSV is missing required columns: {missing_required}")

timestamp_probe = pd.to_datetime(raw["transaction_timestamp"], errors="coerce")
dataset_summary = {
    "rows": int(len(raw)),
    "columns": int(raw.shape[1]),
    "users": int(raw["user_id"].nunique()),
    "duplicate_transaction_ids": int(raw["transaction_id"].duplicated().sum()),
    "invalid_timestamps": int(timestamp_probe.isna().sum()),
    "nonpositive_amounts": int((pd.to_numeric(raw["amount"], errors="coerce") <= 0).sum()),
    "date_min": timestamp_probe.min().isoformat(),
    "date_max": timestamp_probe.max().isoformat(),
    "controlled_anomaly_rows": int(raw["is_anomaly"].notna().sum()),
    "all_synthetic": bool(raw["is_synthetic"].fillna(False).astype(bool).all()),
}
display(pd.Series(dataset_summary, name="value").to_frame())

quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_percent": (100 * raw.isna().mean()).round(2),
    "unique_values": raw.nunique(dropna=True),
}).sort_values(["missing_percent", "unique_values"], ascending=[False, True])
display(quality.head(15))

assert dataset_summary["duplicate_transaction_ids"] == 0, "Transaction IDs must be unique."
assert dataset_summary["invalid_timestamps"] == 0, "Every transaction requires a valid timestamp."
assert dataset_summary["nonpositive_amounts"] == 0, "Transaction amounts must be positive."
print("Core raw-data checks: PASS")


In [ ]:
# Clean a copy; never overwrite the raw dataframe.
data = raw.copy()
date_columns = [
    "transaction_timestamp", "user_date_of_birth",
    "budget_period_start", "budget_period_end",
]
for column in date_columns:
    data[column] = pd.to_datetime(data[column], errors="coerce")
for column in ["amount", "user_monthly_income", "budget_amount"]:
    data[column] = pd.to_numeric(data[column], errors="coerce")

def boolean_values(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.strip().str.lower().isin(
        {"true", "1", "yes", "y", "t"}
    )

data["is_recurring"] = boolean_values(data["is_recurring"])
data["is_anomaly_target"] = boolean_values(data["is_anomaly"]).astype(int)
data["merchant_missing"] = data["merchant"].isna().astype(int)
data["category"] = data["category"].astype(str).str.strip()
data["transaction_type"] = data["transaction_type"].astype(str).str.strip().str.lower()
data = (
    data.drop_duplicates("transaction_id")
    .dropna(subset=["transaction_timestamp", "amount", "user_id", "category"])
)
data = data[data["amount"] > 0].sort_values(
    ["transaction_timestamp", "transaction_id"]
).reset_index(drop=True)
expense = data[data["transaction_type"].eq("expense")].copy()
income = data[data["transaction_type"].eq("income")].copy()

print(f"Processed rows: {len(data):,}")
print(f"Expense rows: {len(expense):,}; income rows: {len(income):,}")
print(f"Controlled anomaly rate among expenses: {100 * expense['is_anomaly_target'].mean():.3f}%")


In [ ]:
category_summary = (
    expense.groupby("category")
    .agg(transactions=("transaction_id", "size"), total_spending_kes=("amount", "sum"))
    .sort_values("total_spending_kes", ascending=False)
)
user_summary = expense.groupby("user_id").agg(
    transactions=("transaction_id", "size"),
    active_days=("transaction_timestamp", lambda values: values.dt.date.nunique()),
    total_spending_kes=("amount", "sum"),
)
display(category_summary)
display(user_summary.describe().round(2))

weekly_total = (
    expense.set_index("transaction_timestamp")["amount"]
    .resample("W-MON")
    .sum()
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
category_summary.sort_values("total_spending_kes")["total_spending_kes"].plot(
    kind="barh", ax=axes[0], color="#2A6F62"
)
axes[0].set_title("Synthetic spending by category")
axes[0].set_xlabel("KES")
weekly_total.plot(ax=axes[1], marker="o", color="#C46A2B")
axes[1].set_title("Total weekly synthetic spending")
axes[1].set_ylabel("KES")
plt.tight_layout()
plt.show()


## 2. Weekly forecasting across every user

Features are built from each user's observed history. Each target is the
next week's expense total. The test period is never used for feature scaling,
model selection, early stopping, or hyperparameter tuning.


In [ ]:
working = data.copy()
working["period_start"] = working["transaction_timestamp"].dt.to_period("W-SUN").dt.start_time
expense_weekly = working[working["transaction_type"].eq("expense")].copy()
income_weekly = working[working["transaction_type"].eq("income")].copy()
expense_weekly["recurring_amount"] = np.where(
    expense_weekly["is_recurring"], expense_weekly["amount"], 0.0
)

base = (
    expense_weekly.groupby(["user_id", "period_start"])
    .agg(
        total_spending=("amount", "sum"),
        transaction_count=("transaction_id", "size"),
        average_transaction_amount=("amount", "mean"),
        recurring_expenses=("recurring_amount", "sum"),
    )
    .reset_index()
)
category = (
    expense_weekly.pivot_table(
        index=["user_id", "period_start"],
        columns="category",
        values="amount",
        aggfunc="sum",
        fill_value=0.0,
    )
    .rename(columns=lambda value: "category_" + str(value).strip().lower().replace(" ", "_"))
    .reset_index()
)
category_columns = sorted(
    column for column in category.columns if column.startswith("category_")
)
income_agg = (
    income_weekly.groupby(["user_id", "period_start"])["amount"]
    .sum().rename("income").reset_index()
)

period_grid = pd.MultiIndex.from_product(
    [
        sorted(data["user_id"].astype(str).unique()),
        pd.date_range(
            working["period_start"].min(),
            working["period_start"].max(),
            freq="W-MON",
        ),
    ],
    names=["user_id", "period_start"],
).to_frame(index=False)
weekly = (
    period_grid.merge(base, how="left", on=["user_id", "period_start"])
    .merge(category, how="left", on=["user_id", "period_start"])
    .merge(income_agg, how="left", on=["user_id", "period_start"])
)
numeric_fill = [
    "total_spending", "transaction_count", "average_transaction_amount",
    "recurring_expenses", "income", *category_columns,
]
weekly[numeric_fill] = weekly[numeric_fill].fillna(0.0)

budget_unique = (
    data.dropna(subset=["budget_amount"])
    .assign(budget_month=lambda frame: frame["transaction_timestamp"].dt.to_period("M").astype(str))
    .drop_duplicates(["user_id", "budget_month", "category"])
)
monthly_budget = (
    budget_unique.groupby(["user_id", "budget_month"])["budget_amount"]
    .sum().rename("monthly_budget").reset_index()
)
weekly["budget_month"] = weekly["period_start"].dt.to_period("M").astype(str)
weekly = weekly.merge(monthly_budget, how="left", on=["user_id", "budget_month"])
weekly["monthly_budget"] = weekly["monthly_budget"].fillna(0.0)
weekly["budget_amount"] = (
    weekly["monthly_budget"] * 7.0 / weekly["period_start"].dt.days_in_month
)
weekly["budget_difference"] = weekly["budget_amount"] - weekly["total_spending"]

weekly = weekly.sort_values(["user_id", "period_start"]).reset_index(drop=True)
grouped = weekly.groupby("user_id", group_keys=False)
weekly["previous_period_spending"] = grouped["total_spending"].shift(1)
weekly["rolling_average_4"] = grouped["total_spending"].transform(
    lambda values: values.shift(1).rolling(4, min_periods=1).mean()
)
previous = weekly["previous_period_spending"]
weekly["spending_growth_rate"] = np.where(
    previous > 0,
    (weekly["total_spending"] - previous) / previous,
    0.0,
)
weekly[["previous_period_spending", "rolling_average_4", "spending_growth_rate"]] = (
    weekly[["previous_period_spending", "rolling_average_4", "spending_growth_rate"]]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

forecast_features = [
    "total_spending", *category_columns, "transaction_count",
    "average_transaction_amount", "income", "recurring_expenses",
    "budget_difference", "previous_period_spending", "rolling_average_4",
    "spending_growth_rate",
]
assert len(forecast_features) == 20, f"Expected 20 features, found {len(forecast_features)}"
assert np.isfinite(weekly[forecast_features].to_numpy(dtype=float)).all()
print(f"Weekly panel: {weekly.shape[0]:,} user-weeks")
print("Forecast features:", forecast_features)


In [ ]:
sequence_x, sequence_y = [], []
sequence_users, input_end_dates, target_dates = [], [], []
for user_id, group in weekly.groupby("user_id", sort=True):
    group = group.sort_values("period_start").reset_index(drop=True)
    values = group[forecast_features].to_numpy(dtype=np.float64)
    targets = group["total_spending"].to_numpy(dtype=np.float64)
    dates = pd.to_datetime(group["period_start"])
    for input_end in range(LOOKBACK - 1, len(group) - FORECAST_HORIZON):
        target_index = input_end + FORECAST_HORIZON
        sequence_x.append(values[input_end - LOOKBACK + 1: input_end + 1])
        sequence_y.append(targets[target_index])
        sequence_users.append(str(user_id))
        input_end_dates.append(dates.iloc[input_end])
        target_dates.append(dates.iloc[target_index])

X_raw = np.stack(sequence_x)
y_raw = np.asarray(sequence_y, dtype=np.float64)
sequence_users = np.asarray(sequence_users)
input_end_dates = pd.to_datetime(input_end_dates).to_numpy()
target_dates = pd.to_datetime(target_dates).to_numpy()
unique_target_dates = np.sort(np.unique(target_dates))
if len(unique_target_dates) < 9:
    raise ValueError("At least nine target weeks are required for chronological evaluation.")

train_date_count = max(1, int(np.floor(len(unique_target_dates) * 0.67)))
validation_end_count = max(train_date_count + 1, int(np.floor(len(unique_target_dates) * 0.84)))
validation_end_count = min(validation_end_count, len(unique_target_dates) - 1)
train_dates = unique_target_dates[:train_date_count]
validation_dates = unique_target_dates[train_date_count:validation_end_count]
test_dates = unique_target_dates[validation_end_count:]
masks = {
    "train": np.isin(target_dates, train_dates),
    "validation": np.isin(target_dates, validation_dates),
    "test": np.isin(target_dates, test_dates),
}
assert all(mask.any() for mask in masks.values())
assert train_dates.max() < validation_dates.min() < test_dates.min()
assert np.all(target_dates > input_end_dates)

train_x_raw = X_raw[masks["train"]]
feature_min = train_x_raw.reshape(-1, train_x_raw.shape[-1]).min(axis=0)
feature_max = train_x_raw.reshape(-1, train_x_raw.shape[-1]).max(axis=0)
feature_scale = np.where(feature_max > feature_min, feature_max - feature_min, 1.0)
train_y_raw = y_raw[masks["train"]]
target_min = float(train_y_raw.min())
target_max = float(train_y_raw.max())
target_scale = target_max - target_min if target_max > target_min else 1.0

forecast_splits = {}
for name, mask in masks.items():
    forecast_splits[name] = {
        "X_raw": X_raw[mask],
        "X": ((X_raw[mask] - feature_min) / feature_scale).astype(np.float32),
        "y_raw": y_raw[mask].astype(np.float64),
        "y": ((y_raw[mask] - target_min) / target_scale).astype(np.float32),
        "user_id": sequence_users[mask],
        "input_end": input_end_dates[mask],
        "target_date": target_dates[mask],
    }

split_table = pd.DataFrame([
    {
        "split": name,
        "rows": len(part["y_raw"]),
        "users": pd.Series(part["user_id"]).nunique(),
        "unique_target_weeks": pd.Series(part["target_date"]).nunique(),
        "target_start": pd.Timestamp(part["target_date"].min()).date(),
        "target_end": pd.Timestamp(part["target_date"].max()).date(),
        "target_mean_kes": part["y_raw"].mean(),
    }
    for name, part in forecast_splits.items()
])
display(split_table.round(2))
print("Training-only scaling check: PASS")
print("Chronological target separation check: PASS")


In [ ]:
def forecast_metric_record(name: str, actual: np.ndarray, predicted: np.ndarray, seconds: float) -> dict:
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)
    absolute_error = np.abs(actual - predicted)
    nonzero = actual != 0
    smape_denominator = np.abs(actual) + np.abs(predicted)
    valid_smape = smape_denominator > 0
    return {
        "model": name,
        "mae_kes": mean_absolute_error(actual, predicted),
        "rmse_kes": mean_squared_error(actual, predicted) ** 0.5,
        "mape_percent": 100 * np.mean(absolute_error[nonzero] / np.abs(actual[nonzero])) if nonzero.any() else np.nan,
        "wape_percent": 100 * absolute_error.sum() / np.abs(actual).sum(),
        "smape_percent": 100 * np.mean(2 * absolute_error[valid_smape] / smape_denominator[valid_smape]),
        "r2": r2_score(actual, predicted),
        "bias_kes": float(np.mean(predicted - actual)),
        "inference_seconds": seconds,
        "evaluated_rows": len(actual),
    }

train_part = forecast_splits["train"]
test_part = forecast_splits["test"]
X_train_flat = train_part["X_raw"].reshape(len(train_part["X_raw"]), -1)
X_test_flat = test_part["X_raw"].reshape(len(test_part["X_raw"]), -1)
y_train = train_part["y_raw"]
y_test = test_part["y_raw"]

baseline_predictions = {}
baseline_models = {}
baseline_times = {}

linear_model = LinearRegression(n_jobs=-1)
linear_model.fit(X_train_flat, y_train)
started = time.perf_counter()
linear_raw_prediction = linear_model.predict(X_test_flat)
baseline_times["linear_regression"] = time.perf_counter() - started
baseline_predictions["linear_regression"] = np.maximum(0.0, linear_raw_prediction)
baseline_models["linear_regression"] = linear_model

random_forest_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_forest_model.fit(X_train_flat, y_train)
started = time.perf_counter()
baseline_predictions["random_forest"] = np.maximum(
    0.0, random_forest_model.predict(X_test_flat)
)
baseline_times["random_forest"] = time.perf_counter() - started
baseline_models["random_forest"] = random_forest_model

total_spending_index = forecast_features.index("total_spending")
started = time.perf_counter()
baseline_predictions["naive_previous_period"] = test_part["X_raw"][:, -1, total_spending_index]
baseline_times["naive_previous_period"] = time.perf_counter() - started
started = time.perf_counter()
baseline_predictions["moving_average_4"] = test_part["X_raw"][:, -4:, total_spending_index].mean(axis=1)
baseline_times["moving_average_4"] = time.perf_counter() - started

preliminary_metrics = pd.DataFrame([
    forecast_metric_record(name, y_test, prediction, baseline_times[name])
    for name, prediction in baseline_predictions.items()
]).sort_values("mae_kes")
display(preliminary_metrics.round(3))
print("Negative Linear Regression predictions clipped to zero:", int((linear_raw_prediction < 0).sum()))


In [ ]:
tf.keras.backend.clear_session()
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(LOOKBACK, len(forecast_features))),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1),
])
lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)
lstm_path = OUTPUT_DIR / "spendly_final_lstm.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, min_delta=1e-5,
        restore_best_weights=True, mode="min",
    ),
    tf.keras.callbacks.ModelCheckpoint(
        lstm_path, monitor="val_loss", save_best_only=True, mode="min"
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]
training_started = time.perf_counter()
lstm_history = lstm_model.fit(
    forecast_splits["train"]["X"],
    forecast_splits["train"]["y"],
    validation_data=(
        forecast_splits["validation"]["X"],
        forecast_splits["validation"]["y"],
    ),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    verbose=2,
    callbacks=callbacks,
)
lstm_training_seconds = time.perf_counter() - training_started
lstm_model = tf.keras.models.load_model(lstm_path)
started = time.perf_counter()
lstm_scaled_prediction = lstm_model.predict(
    forecast_splits["test"]["X"], batch_size=BATCH_SIZE, verbose=0
).reshape(-1)
lstm_inference_seconds = time.perf_counter() - started
lstm_prediction = np.maximum(
    0.0, lstm_scaled_prediction * target_scale + target_min
)
print("Completed epochs:", len(lstm_history.history["loss"]))
print("Best validation loss:", min(lstm_history.history["val_loss"]))
print(f"LSTM training seconds: {lstm_training_seconds:.2f}")


In [ ]:
all_predictions = {**baseline_predictions, "lstm": lstm_prediction}
all_times = {**baseline_times, "lstm": lstm_inference_seconds}
forecast_metrics = pd.DataFrame([
    forecast_metric_record(name, y_test, prediction, all_times[name])
    for name, prediction in all_predictions.items()
]).sort_values("mae_kes").reset_index(drop=True)
display(forecast_metrics.round(3))

forecast_frames = []
base_index = pd.DataFrame({
    "user_id": test_part["user_id"].astype(str),
    "input_end_timestamp": pd.to_datetime(test_part["input_end"]),
    "target_timestamp": pd.to_datetime(test_part["target_date"]),
    "actual_spending": y_test,
})
for name, values in all_predictions.items():
    frame = base_index.copy()
    frame["model"] = name
    frame["predicted_spending"] = values
    frame["absolute_error"] = np.abs(frame["actual_spending"] - values)
    frame["residual"] = frame["actual_spending"] - values
    forecast_frames.append(frame)
forecast_predictions = pd.concat(forecast_frames, ignore_index=True)
per_date_mae = forecast_predictions.pivot_table(
    index="target_timestamp", columns="model", values="absolute_error", aggfunc="mean"
)
display(per_date_mae.round(2))

mean_predictions = forecast_predictions.pivot_table(
    index="target_timestamp", columns="model", values="predicted_spending", aggfunc="mean"
)
mean_actual = base_index.groupby("target_timestamp")["actual_spending"].mean()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(mean_actual.index, mean_actual.values, marker="o", linewidth=3, color="#17332D", label="Actual")
for name in forecast_metrics["model"]:
    axes[0].plot(mean_predictions.index, mean_predictions[name], marker="o", label=name)
axes[0].set_title("Mean weekly spending: actual vs forecast")
axes[0].set_ylabel("KES per user")
axes[0].legend(fontsize=8)

residual_plot = forecast_predictions[forecast_predictions["model"].isin(["lstm", "linear_regression", "random_forest"])]
sns.boxplot(data=residual_plot, x="model", y="residual", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Forecast residuals; positive means underprediction")
axes[1].set_ylabel("Actual − predicted (KES)")
plt.tight_layout()
plt.show()

history_frame = pd.DataFrame(lstm_history.history)
history_frame[["loss", "val_loss"]].plot(figsize=(8, 4), title="LSTM training and validation loss")
plt.xlabel("Epoch")
plt.ylabel("Scaled MSE")
plt.tight_layout()
plt.show()


In [ ]:
best_baseline_name = forecast_metrics.loc[forecast_metrics["model"] != "lstm", "model"].iloc[0]
paired = forecast_predictions[forecast_predictions["model"].isin(["lstm", best_baseline_name])].pivot_table(
    index=["user_id", "target_timestamp"],
    columns="model",
    values="absolute_error",
).dropna()
per_user_difference = (
    paired["lstm"] - paired[best_baseline_name]
).groupby(level="user_id").mean()
bootstrap_rng = np.random.default_rng(RANDOM_STATE)
bootstrap_means = np.array([
    bootstrap_rng.choice(per_user_difference.to_numpy(), size=len(per_user_difference), replace=True).mean()
    for _ in range(2000)
])
bootstrap_ci = np.quantile(bootstrap_means, [0.025, 0.975])
comparison_summary = {
    "best_non_lstm_model": best_baseline_name,
    "mean_lstm_minus_baseline_mae_kes": float(per_user_difference.mean()),
    "bootstrap_95_percent_ci_kes": [float(bootstrap_ci[0]), float(bootstrap_ci[1])],
    "interpretation": (
        "LSTM has higher MAE" if bootstrap_ci[0] > 0
        else "LSTM has lower MAE" if bootstrap_ci[1] < 0
        else "No clear paired MAE difference"
    ),
}
display(pd.Series(comparison_summary, name="value").to_frame())


## 3. Isolation Forest on future expense transactions

Labels are excluded from model inputs. Past-only behavioural features are
computed chronologically. The earliest 70% of expense transactions train
the model, the next slice tunes contamination and the score threshold, and
the final slice is used once for evaluation.


In [ ]:
def rolling_past_features(group: pd.DataFrame) -> pd.DataFrame:
    ordered = group.sort_values(["transaction_timestamp", "transaction_id"]).copy()
    times = ordered["transaction_timestamp"].to_numpy(dtype="datetime64[ns]")
    amounts = ordered["amount"].to_numpy(dtype=float)
    count_7 = np.zeros(len(ordered), dtype=float)
    sum_7 = np.zeros(len(ordered), dtype=float)
    sum_previous_28 = np.zeros(len(ordered), dtype=float)
    queue_7 = deque()
    queue_35 = deque()
    running_7 = 0.0
    running_35 = 0.0
    for index, (timestamp, amount) in enumerate(zip(times, amounts)):
        cutoff_7 = timestamp - np.timedelta64(7, "D")
        cutoff_35 = timestamp - np.timedelta64(35, "D")
        while queue_7 and queue_7[0][0] < cutoff_7:
            running_7 -= queue_7.popleft()[1]
        while queue_35 and queue_35[0][0] < cutoff_35:
            running_35 -= queue_35.popleft()[1]
        count_7[index] = len(queue_7)
        sum_7[index] = running_7
        sum_previous_28[index] = max(0.0, running_35 - running_7)
        queue_7.append((timestamp, amount))
        queue_35.append((timestamp, amount))
        running_7 += amount
        running_35 += amount
    ordered["transaction_count_last_7d"] = count_7
    comparable_week = sum_previous_28 / 4.0
    recent_change = np.zeros(len(ordered), dtype=float)
    np.divide(sum_7, comparable_week, out=recent_change, where=comparable_week > 0)
    recent_change[comparable_week > 0] -= 1.0
    ordered["recent_spending_change"] = recent_change
    return ordered

anomaly_frame = expense.sort_values(
    ["user_id", "transaction_timestamp", "transaction_id"]
).copy()
anomaly_frame["time_since_previous_hours"] = (
    anomaly_frame.groupby("user_id")["transaction_timestamp"].diff().dt.total_seconds() / 3600
)
anomaly_frame["historical_average_amount"] = anomaly_frame.groupby("user_id")["amount"].transform(
    lambda values: values.shift(1).expanding(min_periods=1).mean()
)
anomaly_frame["deviation_from_historical_average"] = np.where(
    anomaly_frame["historical_average_amount"] > 0,
    anomaly_frame["amount"] / anomaly_frame["historical_average_amount"] - 1.0,
    0.0,
)
anomaly_frame["past_user_count"] = anomaly_frame.groupby("user_id").cumcount()
anomaly_frame["past_category_count"] = anomaly_frame.groupby(["user_id", "category"]).cumcount()
anomaly_frame["category_proportion"] = np.where(
    anomaly_frame["past_user_count"] > 0,
    anomaly_frame["past_category_count"] / anomaly_frame["past_user_count"],
    0.0,
)
anomaly_frame = pd.concat(
    [rolling_past_features(group) for _, group in anomaly_frame.groupby("user_id", sort=False)],
    ignore_index=True,
)
anomaly_frame["hour_of_day"] = anomaly_frame["transaction_timestamp"].dt.hour
anomaly_frame["is_weekend"] = anomaly_frame["transaction_timestamp"].dt.dayofweek.ge(5).astype(int)

unique_times = np.sort(anomaly_frame["transaction_timestamp"].unique())
train_time_count = max(1, int(np.floor(len(unique_times) * 0.70)))
anomaly_train_end = unique_times[train_time_count - 1]
anomaly_train = anomaly_frame[anomaly_frame["transaction_timestamp"] <= anomaly_train_end].copy()
anomaly_held_out = anomaly_frame[anomaly_frame["transaction_timestamp"] > anomaly_train_end].copy()
held_out_times = np.sort(anomaly_held_out["transaction_timestamp"].unique())
validation_time_count = max(1, int(np.floor(len(held_out_times) * 0.20)))
anomaly_validation_end = held_out_times[validation_time_count - 1]
anomaly_validation = anomaly_held_out[
    anomaly_held_out["transaction_timestamp"] <= anomaly_validation_end
].copy()
anomaly_evaluation = anomaly_held_out[
    anomaly_held_out["transaction_timestamp"] > anomaly_validation_end
].copy()

category_mapping = {
    category: index
    for index, category in enumerate(sorted(anomaly_train["category"].astype(str).unique()))
}
for frame in [anomaly_train, anomaly_validation, anomaly_evaluation]:
    frame["category_code"] = frame["category"].map(category_mapping).fillna(-1).astype(int)

anomaly_features = [
    "amount", "category_code", "transaction_count_last_7d",
    "time_since_previous_hours", "deviation_from_historical_average",
    "recent_spending_change", "category_proportion", "is_recurring",
    "hour_of_day", "is_weekend",
]
for frame in [anomaly_train, anomaly_validation, anomaly_evaluation]:
    frame[anomaly_features] = frame[anomaly_features].replace([np.inf, -np.inf], np.nan).fillna(0.0)

anomaly_split_table = pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "users": frame["user_id"].nunique(),
        "start": frame["transaction_timestamp"].min(),
        "end": frame["transaction_timestamp"].max(),
        "labelled_anomalies": int(frame["is_anomaly_target"].sum()),
    }
    for name, frame in [
        ("train", anomaly_train),
        ("validation", anomaly_validation),
        ("evaluation", anomaly_evaluation),
    ]
])
display(anomaly_split_table)
assert anomaly_train["transaction_timestamp"].max() < anomaly_validation["transaction_timestamp"].min()
assert anomaly_validation["transaction_timestamp"].max() < anomaly_evaluation["transaction_timestamp"].min()
print("Isolation Forest chronological separation check: PASS")


In [ ]:
def anomaly_metric_record(actual: np.ndarray, predicted: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="binary", zero_division=0
    )
    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "false_positive_rate": float(fp / (fp + tn)) if fp + tn else 0.0,
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn),
        "predicted_anomalies": int(predicted.sum()),
        "labelled_anomalies": int(actual.sum()),
        "rows": int(len(actual)),
    }

anomaly_preprocessor = RobustScaler()
anomaly_train_values = anomaly_preprocessor.fit_transform(anomaly_train[anomaly_features])
anomaly_validation_values = anomaly_preprocessor.transform(anomaly_validation[anomaly_features])
anomaly_evaluation_values = anomaly_preprocessor.transform(anomaly_evaluation[anomaly_features])
validation_labels = anomaly_validation["is_anomaly_target"].to_numpy(dtype=bool)
evaluation_labels = anomaly_evaluation["is_anomaly_target"].to_numpy(dtype=bool)

candidate_contamination = [0.005, 0.010, 0.015, 0.020, 0.030]
candidate_models = {}
tuning_rows = []
for contamination in candidate_contamination:
    candidate = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    candidate.fit(anomaly_train_values)
    candidate_models[contamination] = candidate
    default_prediction = candidate.predict(anomaly_validation_values) == -1
    tuning_rows.append({
        "contamination": contamination,
        **anomaly_metric_record(validation_labels, default_prediction),
    })
anomaly_tuning = pd.DataFrame(tuning_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
display(anomaly_tuning[[
    "contamination", "precision", "recall", "f1",
    "false_positive_rate", "predicted_anomalies",
]].round(4))

selected_contamination = float(anomaly_tuning.iloc[0]["contamination"])
anomaly_model = candidate_models[selected_contamination]
validation_scores = -anomaly_model.decision_function(anomaly_validation_values)
pr_precision, pr_recall, thresholds = precision_recall_curve(validation_labels, validation_scores)
if len(thresholds):
    threshold_f1 = 2 * pr_precision[:-1] * pr_recall[:-1] / np.maximum(
        pr_precision[:-1] + pr_recall[:-1], 1e-12
    )
    threshold_index = int(np.nanargmax(threshold_f1))
    anomaly_threshold = float(thresholds[threshold_index])
else:
    anomaly_threshold = float(np.quantile(validation_scores, 1 - selected_contamination))

validation_prediction = validation_scores >= anomaly_threshold
validation_threshold_metrics = anomaly_metric_record(validation_labels, validation_prediction)
evaluation_scores = -anomaly_model.decision_function(anomaly_evaluation_values)
evaluation_prediction = evaluation_scores >= anomaly_threshold
anomaly_evaluation_metrics = anomaly_metric_record(evaluation_labels, evaluation_prediction)
anomaly_evaluation_metrics.update({
    "selected_contamination": selected_contamination,
    "decision_threshold": anomaly_threshold,
})
display(pd.Series(anomaly_evaluation_metrics, name="evaluation").to_frame())

anomaly_results = anomaly_evaluation[[
    "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
    "anomaly_type", "is_anomaly_target",
]].copy()
anomaly_results["anomaly_score"] = evaluation_scores
anomaly_results["predicted_anomaly"] = evaluation_prediction
scenario_metrics = (
    anomaly_results[anomaly_results["is_anomaly_target"].eq(1)]
    .groupby("anomaly_type")
    .agg(labelled=("is_anomaly_target", "size"), detected=("predicted_anomaly", "sum"))
    .assign(recall=lambda frame: frame["detected"] / frame["labelled"])
    .sort_values("recall")
)
alert_categories = (
    anomaly_results[anomaly_results["predicted_anomaly"]]
    .groupby("category")
    .agg(alerts=("transaction_id", "size"), true_anomalies=("is_anomaly_target", "sum"))
    .sort_values("alerts", ascending=False)
)
display(scenario_metrics.round(4))
display(alert_categories)


In [ ]:
matrix = confusion_matrix(
    anomaly_results["is_anomaly_target"],
    anomaly_results["predicted_anomaly"],
    labels=[0, 1],
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"], ax=axes[0]
)
axes[0].set_title("Isolation Forest confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Controlled label")

sns.histplot(
    data=anomaly_results,
    x="anomaly_score",
    hue="is_anomaly_target",
    bins=50,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[1],
)
axes[1].axvline(anomaly_threshold, color="red", linestyle="--", label="Threshold")
axes[1].set_title("Anomaly-score distributions")
axes[1].legend(title="Label")
plt.tight_layout()
plt.show()

print("Highest-scoring future transactions")
display(
    anomaly_results.nlargest(15, "anomaly_score")[[
        "transaction_timestamp", "user_id", "amount", "category",
        "anomaly_score", "is_anomaly_target", "predicted_anomaly", "anomaly_type",
    ]]
)


## 4. Save evidence and print the handoff block

These files are notebook-only outputs. Do not copy them into the application
until the results have been reviewed.


In [ ]:
forecast_metrics.to_csv(OUTPUT_DIR / "forecast_metrics.csv", index=False)
forecast_predictions.to_csv(OUTPUT_DIR / "forecast_predictions.csv", index=False)
anomaly_tuning.to_csv(OUTPUT_DIR / "anomaly_validation_tuning.csv", index=False)
anomaly_results.to_csv(OUTPUT_DIR / "anomaly_evaluation_predictions.csv", index=False)
scenario_metrics.to_csv(OUTPUT_DIR / "anomaly_scenario_metrics.csv")
alert_categories.to_csv(OUTPUT_DIR / "anomaly_alert_categories.csv")
split_table.to_csv(OUTPUT_DIR / "forecast_split_summary.csv", index=False)
anomaly_split_table.to_csv(OUTPUT_DIR / "anomaly_split_summary.csv", index=False)

joblib.dump(linear_model, OUTPUT_DIR / "linear_regression.joblib")
joblib.dump(random_forest_model, OUTPUT_DIR / "random_forest.joblib")
joblib.dump(anomaly_model, OUTPUT_DIR / "isolation_forest.joblib")
joblib.dump(anomaly_preprocessor, OUTPUT_DIR / "anomaly_preprocessor.joblib")
joblib.dump({
    "feature_min": feature_min,
    "feature_max": feature_max,
    "feature_scale": feature_scale,
    "target_min": target_min,
    "target_max": target_max,
    "target_scale": target_scale,
}, OUTPUT_DIR / "forecast_scaler.joblib")

feature_schema = {
    "forecast_features": forecast_features,
    "lookback": LOOKBACK,
    "forecast_horizon": FORECAST_HORIZON,
    "anomaly_features": anomaly_features,
    "anomaly_category_mapping": category_mapping,
}
(OUTPUT_DIR / "feature_schema.json").write_text(
    json.dumps(feature_schema, indent=2), encoding="utf-8"
)

final_payload = {
    "notebook": "Spending_Model_Final_Evaluation",
    "dataset_sha256": dataset_sha256,
    "dataset": dataset_summary,
    "forecast_design": {
        "lookback_weeks": LOOKBACK,
        "horizon_weeks": FORECAST_HORIZON,
        "features": len(forecast_features),
        "split_summary": split_table.astype(str).to_dict(orient="records"),
        "scaler_fit": "training_only",
    },
    "forecast_metrics": forecast_metrics.round(6).to_dict(orient="records"),
    "forecast_comparison": comparison_summary,
    "lstm": {
        "epochs_completed": len(lstm_history.history["loss"]),
        "best_validation_loss": float(min(lstm_history.history["val_loss"])),
        "training_seconds": float(lstm_training_seconds),
    },
    "anomaly_design": {
        "features": anomaly_features,
        "split_summary": anomaly_split_table.astype(str).to_dict(orient="records"),
        "labels_used_as_features": False,
        "preprocessor_fit": "training_only",
    },
    "anomaly_validation_threshold_metrics": validation_threshold_metrics,
    "anomaly_evaluation_metrics": anomaly_evaluation_metrics,
    "anomaly_scenario_metrics": scenario_metrics.reset_index().to_dict(orient="records"),
    "checks": {
        "raw_transaction_ids_unique": dataset_summary["duplicate_transaction_ids"] == 0,
        "strict_forecast_chronology": True,
        "strict_anomaly_chronology": True,
        "training_only_scaling": True,
        "test_used_for_model_selection": False,
    },
}
(OUTPUT_DIR / "spendly_final_results.json").write_text(
    json.dumps(final_payload, indent=2, default=str), encoding="utf-8"
)

zip_path = Path("spendly_final_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print(f"Saved results folder: {OUTPUT_DIR.resolve()}")
print(f"Saved downloadable bundle: {zip_path.resolve()}")


In [ ]:
print("=" * 24 + " COPY RESULTS FROM HERE " + "=" * 24)
print(json.dumps(final_payload, indent=2, default=str))
print("=" * 27 + " END RESULTS " + "=" * 27)


### What to send back

Copy the complete text between **COPY RESULTS FROM HERE** and
**END RESULTS** from the cell above. If the notebook stops with an error,
send the full error message and the number of the cell that failed.

Do not replace application models yet; review these benchmark results first.
